<a href="https://colab.research.google.com/github/evipt/nlp-course/blob/main/week40.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sequence Labelling for Cross-Lingual Question Answering

This notebook demonstrates how to train token‐classification models for extractive question answering using the TyDi QA–based XOR RC dataset. It follows the same approach as the provided `sequence_labeller.py` script.

We will:

1. Load the dataset and select examples by language (Arabic, Korean, Telugu).
2. Tokenize questions and contexts using a multilingual transformer (e.g. XLM–RoBERTa).
3. Align character‐level answer spans to token‐level BIO labels.
4. Train and evaluate a token‐classification model for each language.

If a question is unanswerable, all context tokens are labelled as `O`.

**Note:** You need to install the `datasets`, `transformers` and `seqeval` packages to run this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install required libraries if not already installed.
# Uncomment the following lines if running in a new environment.
!pip install datasets transformers seqeval -q

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments
)
from seqeval.metrics import precision_score, recall_score, f1_score

# Suppress long text wrapping
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def prepare_features(examples, tokenizer, max_length=512, label_all_tokens=False):
    """Tokenize a batch of examples and align answer spans to BIO labels.
    Parameters:
        examples: dict with keys 'question', 'context' and 'answers'.
        tokenizer: a fast HuggingFace tokenizer.
        max_length: maximum sequence length.
        label_all_tokens: whether to label every sub‑token inside an answer span.
    Returns:
        dict with tokenized inputs and aligned labels.
    """
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=max_length,
        padding='max_length',
        return_offsets_mapping=True,
        return_token_type_ids=True,
    )
    labels = []
    for i, offsets in enumerate(tokenized['offset_mapping']):
        seq_ids = tokenized.sequence_ids(i)
        label_ids = [-100] * len(offsets)
        answer_starts = examples['answers'][i]['answer_start'] if examples['answers'][i] else []
        answer_texts = examples['answers'][i]['text'] if examples['answers'][i] else []
        if len(answer_starts) == 0 or len(answer_texts) == 0:
            # Unanswerable: mark context tokens as 'O' (2)
            for j, (start_offset, end_offset) in enumerate(offsets):
                if seq_ids[j] == 1 and start_offset != end_offset:
                    label_ids[j] = 2
            labels.append(label_ids)
            continue
        answer_start = answer_starts[0]
        answer_text = answer_texts[0]
        answer_end = answer_start + len(answer_text)
        answer_started = False
        for j, (start_offset, end_offset) in enumerate(offsets):
            if seq_ids[j] != 1 or start_offset == end_offset:
                label_ids[j] = -100
                continue
            if end_offset <= answer_start or start_offset >= answer_end:
                label_ids[j] = 2
                answer_started = False
                continue
            if not answer_started:
                label_ids[j] = 0
                answer_started = True
            else:
                label_ids[j] = 1
                if not label_all_tokens:
                    answer_started = True
        labels.append(label_ids)
    tokenized['labels'] = labels
    return tokenized

In [ ]:
def compute_metrics(p):
    predictions, label_ids = p
    preds = np.argmax(predictions, axis=2)
    label_list = ['B-ANS', 'I-ANS', 'O']
    true_labels = []
    true_preds = []
    for pred, lab in zip(preds, label_ids):
        pred_labels = []
        lab_labels = []
        for p_i, l_i in zip(pred, lab):
            if l_i == -100:
                continue
            pred_labels.append(label_list[p_i])
            lab_labels.append(label_list[l_i])
        true_preds.append(pred_labels)
        true_labels.append(lab_labels)
    precision = precision_score(true_labels, true_preds)
    recall = recall_score(true_labels, true_preds)
    f1 = f1_score(true_labels, true_preds)
    return {
        'precision': precision,
        'recall': recall,
        'f1': f1
    }

In [ ]:
def prepare_dataset_for_language(dataset, language_code):
    """Filter and transform the dataset for a single language.
    Each example is converted to a format with 'question', 'context' and
    'answers' fields.
    """
    def add_answers(example):
        answerable_flag = example.get('answerable', example.get('answertable', True))
        has_answer = (
            bool(answerable_flag)
            and example.get('answer_start', -1) is not None
            and int(example.get('answer_start', -1)) >= 0
            and example.get('answer', '') not in ('', 'no', 'yes')
        )
        if has_answer:
            return {
                'question': example['question'],
                'context': example['context'],
                'answers': {
                    'text': [example['answer']],
                    'answer_start': [int(example['answer_start'])]
                }
            }
        else:
            return {
                'question': example['question'],
                'context': example['context'],
                'answers': {
                    'text': [],
                    'answer_start': []
                }
            }

    train_ds = dataset['train'].filter(lambda ex: ex.get('lang', ex.get('language')) == language_code)
    val_ds = dataset['validation'].filter(lambda ex: ex.get('lang', ex.get('language')) == language_code)
    train_ds = train_ds.map(add_answers, remove_columns=train_ds.column_names)
    val_ds = val_ds.map(add_answers, remove_columns=val_ds.column_names)
    return train_ds, val_ds

In [17]:
def train_language_model(train_ds, val_ds, model_name='xlm-roberta-base', epochs=3, batch_size=4, max_length=512):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    def _prepare(examples):
        return prepare_features(examples, tokenizer=tokenizer, max_length=max_length)
    tokenized_train = train_ds.map(_prepare, batched=True, remove_columns=train_ds.column_names)
    tokenized_val = val_ds.map(_prepare, batched=True, remove_columns=val_ds.column_names)
    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels=3,
        id2label={0: 'B-ANS', 1: 'I-ANS', 2: 'O'},
        label2id={'B-ANS': 0, 'I-ANS': 1, 'O': 2}
    )
    data_collator = DataCollatorForTokenClassification(tokenizer)
    training_args = TrainingArguments(
        output_dir=f'/content/drive/MyDrive/qa_project/final_model_3/{lang}',
        eval_strategy='epoch',
        save_strategy='epoch',
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        logging_steps=50,
        learning_rate=3e-5,
        weight_decay=0.01,
        warmup_ratio=0.05,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model='f1'
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )
    trainer.train()
    metrics = trainer.evaluate()
    return trainer, metrics

In [18]:
# Example usage: Load the dataset and train on one language
# Uncomment and run the following lines to train on Arabic, Korean and Telugu.

dataset = load_dataset('coastalcph/tydi_xor_rc')
languages = ['ar', 'ko', 'te']
results = {}
for lang in languages:
    train_ds, val_ds = prepare_dataset_for_language(dataset, lang)
    trainer, metrics = train_language_model(train_ds, val_ds, epochs=3, batch_size=4)
    results[lang] = metrics
    print(f'Validation metrics for {lang}:', metrics)

print('Final comparison:', results)

Map:   0%|          | 0/2558 [00:00<?, ? examples/s]

Map:   0%|          | 0/415 [00:00<?, ? examples/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.147000,0.113670,0.173333,0.035813,0.059361
2,0.132800,0.102054,0.334204,0.352617,0.343164
3,0.072100,0.111900,0.325926,0.363636,0.343750


Validation metrics for ar: {'eval_loss': 0.1119004487991333, 'eval_precision': 0.32592592592592595, 'eval_recall': 0.36363636363636365, 'eval_f1': 0.34375000000000006, 'eval_runtime': 12.6606, 'eval_samples_per_second': 32.779, 'eval_steps_per_second': 8.214, 'epoch': 3.0}


...

Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.131900,0.087155,0.162162,0.020619,0.036585


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.131900,0.087155,0.162162,0.020619,0.036585
2,0.104800,0.091727,0.194726,0.329897,0.244898
3,0.078400,0.087592,0.263158,0.257732,0.260417


Validation metrics for te: {'eval_loss': 0.08759228140115738, 'eval_precision': 0.2631578947368421, 'eval_recall': 0.25773195876288657, 'eval_f1': 0.2604166666666667, 'eval_runtime': 11.7277, 'eval_samples_per_second': 32.743, 'eval_steps_per_second': 8.186, 'epoch': 3.0}
Final comparison: {'ar': {'eval_loss': 0.1119004487991333, 'eval_precision': 0.32592592592592595, 'eval_recall': 0.36363636363636365, 'eval_f1': 0.34375000000000006, 'eval_runtime': 12.6606, 'eval_samples_per_second': 32.779, 'eval_steps_per_second': 8.214, 'epoch': 3.0}, 'ko': {'eval_loss': 0.11219660192728043, 'eval_precision': 0.41847826086956524, 'eval_recall': 0.456973293768546, 'eval_f1': 0.4368794326241135, 'eval_runtime': 11.6027, 'eval_samples_per_second': 30.682, 'eval_steps_per_second': 7.671, 'epoch': 3.0}, 'te': {'eval_loss': 0.08759228140115738, 'eval_precision': 0.2631578947368421, 'eval_recall': 0.25773195876288657, 'eval_f1': 0.2604166666666667, 'eval_runtime': 11.7277, 'eval_samples_per_second': 32.7

In [19]:
import numpy as np

def preprocess_test(examples, tokenizer, max_length=512):
    # tokenize question/context pairs without labels
    tokenized = tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=max_length,
        padding='max_length',
        return_offsets_mapping=True,
        return_token_type_ids=True,
    )
    return tokenized

def predict_on_test(trainer, tokenizer, test_ds, max_length=512):
    # tokenize test examples
    tokenized_test = test_ds.map(
        lambda x: preprocess_test(x, tokenizer, max_length),
        batched=True,
        remove_columns=test_ds.column_names
    )
    # run the model
    outputs = trainer.predict(tokenized_test)
    pred_ids = np.argmax(outputs.predictions, axis=2)
    label_list = ['B-ANS', 'I-ANS', 'O']

    answers = []
    for i, preds in enumerate(pred_ids):
        offsets = tokenized_test[i]['offset_mapping']
        context = test_ds[i]['context']
        ans_tokens = []

        # accumulate answer span based on BIO labels
        start_char, end_char = None, None
        for p_id, (start, end) in zip(preds, offsets):
            label = label_list[p_id]
            if label == 'B-ANS':
                if start_char is not None:
                    ans_tokens.append(context[start_char:end_char])
                start_char, end_char = start, end
            elif label == 'I-ANS' and start_char is not None:
                end_char = end
            else:
                if start_char is not None:
                    ans_tokens.append(context[start_char:end_char])
                    start_char = end_char = None
        # catch any trailing span
        if start_char is not None:
            ans_tokens.append(context[start_char:end_char])

        # join multiple answer tokens if needed
        answers.append(" ".join(ans_tokens))
    return answers

# example usage:
# test_answers = predict_on_test(trainer, tokenizer, your_test_dataset)
# print(test_answers[:5])

In [20]:
# example usage:

test_ds = dataset['test']
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base', use_fast=True)


test_answers = predict_on_test(trainer, tokenizer, test_ds)
print(test_answers)

['Paris', 'William Shakespeare', 'Mount Everest', 'Sydney']
